## Silver Stage 2: SCD Type 2 Dimensions
Builds `dim_customer_scd2` and `dim_product_scd2` with full change history. Initial load comes from the cleaned batch data; then each day's CDC file is replayed in date order using a two-step MERGE (close old row, insert new row).

In [0]:
%run "./00_setup"

%md
### Step 0: Setup
Creating catalog, schemas (raw/silver/gold), and a volume to store raw files.

Catalog and schemas created successfully


%md
### Helper: Row Hash
Concatenates tracked columns into one hash so we can detect a change with a single comparison instead of checking every column individually.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

FAR_FUTURE = "9999-12-31"

def row_hash(cols):
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NA")) for c in cols]), 256)

%md
### Initial Load: Customer Dimension
Every cleaned customer starts as one "current" row. `effective_start_date` = signup_date, `effective_end_date` = far future, `is_current` = true. Surrogate key is a hash of customer_id + start date.

In [0]:
customer_tracked_cols = ["customer_name", "city", "segment", "gender", "status"]

customers_clean = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean")

dim_customer_init = (
    customers_clean
    .withColumn("hash_value", row_hash(customer_tracked_cols))
    .withColumn("effective_start_date", F.col("signup_date"))
    .withColumn("effective_end_date", F.to_date(F.lit(FAR_FUTURE)))
    .withColumn("is_current", F.lit(True))
    .withColumn("customer_sk", F.sha2(F.concat_ws("||", "customer_id", F.col("effective_start_date").cast("string")), 256))
    .select("customer_sk", "customer_id", "customer_name", "city", "segment", "gender", "status",
            "signup_date", "effective_start_date", "effective_end_date", "is_current", "hash_value")
)

dim_customer_init.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2")

print("initial customer dim rows:", dim_customer_init.count())

initial customer dim rows: 2500


%md
### Initial Load: Product Dimension
Same pattern as customers — starting state for every product, tracked columns hashed for future change detection.

In [0]:
product_tracked_cols = ["product_name", "category", "brand", "unit_price", "status"]

products_clean = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver1_products_clean")

dim_product_init = (
    products_clean
    .withColumn("hash_value", row_hash(product_tracked_cols))
    .withColumn("effective_start_date", F.col("created_date"))
    .withColumn("effective_end_date", F.to_date(F.lit(FAR_FUTURE)))
    .withColumn("is_current", F.lit(True))
    .withColumn("product_sk", F.sha2(F.concat_ws("||", "product_id", F.col("effective_start_date").cast("string")), 256))
    .select("product_sk", "product_id", "product_name", "category", "brand", "unit_price", "status",
            "created_date", "effective_start_date", "effective_end_date", "is_current", "hash_value")
)

dim_product_init.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2")

print("initial product dim rows:", dim_product_init.count())

initial product dim rows: 780


%md
### SCD2 Apply Function
For each day's CDC batch (oldest first), compares hash of incoming record vs the current active row. If different (or new), closes the old row (Step 1) and inserts a new current row with a fresh surrogate key (Step 2). Processing by effective_date keeps history correct even with late-arriving data.

In [0]:
def apply_scd2(target_table_name, cdc_df, business_key, tracked_cols, sk_col):
    target = DeltaTable.forName(spark, target_table_name)

    cdc_df = cdc_df.withColumn("hash_value", row_hash(tracked_cols))
    dates = [r[0] for r in cdc_df.select("effective_date").distinct().orderBy("effective_date").collect()]

    for eff_date in dates:
        day_batch = cdc_df.filter(F.col("effective_date") == eff_date)

        current_df = target.toDF().filter("is_current = true")
        changed = (
            day_batch.alias("s")
            .join(
                current_df.alias("t"),
                F.col(f"s.{business_key}") == F.col(f"t.{business_key}"),
                "left"
            )
            .filter(F.col(f"t.{business_key}").isNull() | (F.col("s.hash_value") != F.col("t.hash_value")))
            .select("s.*")
        )

        if changed.limit(1).count() == 0:
            continue

        (target.alias("t")
            .merge(changed.alias("s"), f"t.{business_key} = s.{business_key} AND t.is_current = true")
            .whenMatchedUpdate(set={
                "is_current": "false",
                "effective_end_date": "date_sub(s.effective_date, 1)"
            })
            .execute())

        new_rows = (
            changed
            .withColumn("effective_start_date", F.col("effective_date"))
            .withColumn("effective_end_date", F.to_date(F.lit(FAR_FUTURE)))
            .withColumn("is_current", F.lit(True))
            .withColumn(sk_col, F.sha2(F.concat_ws("||", business_key, F.col("effective_start_date").cast("string")), 256))
        )

        target_cols = target.toDF().columns
        new_rows = new_rows.select(*target_cols)

        (target.alias("t")
            .merge(new_rows.alias("s"), f"t.{sk_col} = s.{sk_col}")
            .whenNotMatchedInsertAll()
            .execute())

        print(f"{target_table_name}: applied {changed.count()} changes for effective_date={eff_date}")

%md
### Apply Customer CDC
Reading the customer CDC bronze table, deduping to one change per key per day, then applying SCD2 logic.

In [0]:
customers_cdc = (
    spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_cdc")
    .withColumn("effective_date", F.expr("try_to_date(effective_date)"))
    .withColumn("signup_date", F.expr("try_to_date(signup_date, 'yyyy-MM-dd')"))
    .dropDuplicates(["customer_id", "effective_date"])
    .filter(F.col("effective_date").isNotNull())
)

apply_scd2(
    target_table_name=f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2",
    cdc_df=customers_cdc,
    business_key="customer_id",
    tracked_cols=customer_tracked_cols,
    sk_col="customer_sk",
)

retail_demo.silver.dim_customer_scd2: applied 0 changes for effective_date=2026-04-24
retail_demo.silver.dim_customer_scd2: applied 0 changes for effective_date=2026-04-25
retail_demo.silver.dim_customer_scd2: applied 0 changes for effective_date=2026-04-26


%md
### Apply Product CDC
Same pattern, plus casting unit_price back to double since it comes in as string from bronze.

In [0]:
products_cdc = (
    spark.table(f"{CATALOG}.{RAW_SCHEMA}.bronze_products_cdc")
    .withColumn("effective_date", F.expr("try_to_date(effective_date)"))
    .withColumn("created_date", F.expr("try_to_date(created_date, 'yyyy-MM-dd')"))
    .withColumn("unit_price", F.expr("try_cast(unit_price AS DOUBLE)"))
    .dropDuplicates(["product_id", "effective_date"])
    .filter(F.col("effective_date").isNotNull())
)

apply_scd2(
    target_table_name=f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2",
    cdc_df=products_cdc,
    business_key="product_id",
    tracked_cols=product_tracked_cols,
    sk_col="product_sk",
)

retail_demo.silver.dim_product_scd2: applied 0 changes for effective_date=2026-04-24
retail_demo.silver.dim_product_scd2: applied 0 changes for effective_date=2026-04-25
retail_demo.silver.dim_product_scd2: applied 0 changes for effective_date=2026-04-26


%md
### Sanity Check: Confirm SCD2 History Is Being Tracked
Find customers/products with more than 1 historical row — proves the two-step MERGE is actually creating new versions, not just overwriting.

In [0]:
print("Total customer dim rows:", spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2").count())
print("Total product dim rows:", spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_product_scd2").count())

display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_scd2")
    .groupBy("customer_id").count().filter("count > 1")
)

Total customer dim rows: 3093
Total product dim rows: 960


customer_id,count
C00175,2
C00552,3
C00929,3
C01257,2
C01414,2
C01426,2
C01684,2
C01937,2
C02069,2
C02309,3
